# MODELO ESTIMACIÓN PROBABILIDAD DE CONTINUACIÓN FUTUROS CRIPTOMONEDAS - BINANCE

In [45]:
# =========================================================
# LIBRERÍAS
# =========================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

from ta.trend import EMAIndicator, ADXIndicator, MACD
from ta.momentum import RSIIndicator, ROCIndicator
from ta.volatility import AverageTrueRange

from sklearn.metrics import precision_score, confusion_matrix
import lightgbm as lgb
from lightgbm import LGBMClassifier

from dateutil.relativedelta import relativedelta
import joblib

import warnings
warnings.filterwarnings("ignore")

import winsound
import time

# 0. ENTRENAMIENTO DEL MODELO

## 🎯 Objetivo del modelo

Detectar reversiones de sobrecompra / sobreventa en 1H, con resultado en en al menos MIN_HOLD_BARS velas de 15 minutos.

Reversiones fuertes en al menos MIN_HOLD_BARS velas de 15 minutos, con confirmación posterior en 15m.

👉 Esto implica que NO necesitas años de datos, necesitas suficientes ciclos completos de mercado.

Función y Propósito

get_all_usdt_futures_symnbols: Obtiene el universo de símbolos sobre los que el modelo puede operar.

filter_symbols_pre_ml: Filtra símbolos antes del scanner ML. Selecciona criptos líquidas, volátiles y operables para reversiones duraderas.

get_ohlc: Descargar velas OHLCV limpias y confiables desde Binance.

add_features: Calcular features de ML coherentes con el entrenamiento.

build_dataset: Construir el dataset de entrenamiento del modelo ML.

train_model: Entrenar el modelo ML de probabilidad de reversión.

scan_market_ml: Detectar símbolos candidatos a reversión (1H).

df15_provider: Proveer velas 15m con suficiente contexto para análisis secuencial.

atr_15m_provider: Calcular ATR realista y estable para gestión de riesgo.

confirm_entry_15m_realtime: Esperar y confirmar el momento exacto de entrada en velas de 15 minutos.

structural_exit_conditions: Detectar cuando el contexto de reversión deja de ser válido.

monitor_trade: Gestionar TODO el ciclo de vida del trade con alertas.


In [46]:
# =========================================================
# CONFIGURACIÓN MODELO
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"

DATA_SOURCE = "FILE" # Fuente de datos de entrenamiento FILE para csv almacenado en disco duro, o BINANCE para datos más recientes descargados desde Binance
DATASET_FILE = "Dataset_continuacion_futuros_cripto_SPLIT6-2_2026-02-08_1504" + ".parquet"
DATASET_PATH = "C:/Users/Lenovo S540/Python_scripts/DATASETS_FUTURES_CONTINUATION/" + DATASET_FILE

TRAIN_WEEKS = 6
TEST_WEEKS = 2
FOLDS = 5
STEP = 1 # Sliding window step IN WEEKS for each split

#TRAIN_WEEKS = 2
#TEST_WEEKS = 1
#FOLDS = 1
#STEP = 1 # Sliding window step IN WEEKS for each split

MODEL_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.8,
    "random_state": 42
}

TARGET = "target"

FEATURES = [
    "ema_20_slope",
    "ema_50_slope",
    "ema_ratio_20_50",
    "adx",
    "di_plus",
    "di_minus",
    "atr_ratio",
    "roc_10",
    "roc_20",
    "vol_zscore",
    "vol_ema_ratio",
    "body_range_ratio",
    "close_pos",
    "range_contraction"
]

PROBA_THRESHOLD = 0.65     # umbral operativo (no afecta entrenamiento)
NEG_DOWNSAMPLE_FRAC = 0.25


# ======================
# CONFIGURACIÓN SCANNER
# ======================
CONFIDENCE_THRESHOLD = 0.65     # confianza mínima del modelo
CONFIDENCE_THRESHOLD = 0.2

LOOKBACK_15M = 120              # velas 15m para features
LOOKBACK_5M = 60                # velas 5m para confirmación
SCAN_INTERVAL_SECONDS = 60      # loop principal (1 min)

TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"


# ======================
# CONFIGURACIÓN GESTIÓN DE RIESGO
# ======================
VOL_MULT = 1.2     # Multiplicador de volumen como condición de entrada de la orden en timeframe 5 minutos
TRADE_CHECK_INTERVAL = 30  # segundos
TP1_ATR = 1.0
TP2_ATR = 2.5
SL_ATR = 1.2
TRAIL_BUFFER_ATR = 0.3


| Feature              | Rol en continuaciones   |
| -------------------- | ----------------------- |
| `ema_20_slope`       | momentum corto          |
| `ema_50_slope`       | momentum estructural    |
| `ema_ratio_20_50`    | alineación de tendencia |
| `adx`                | fuerza del régimen      |
| `di_plus / di_minus` | dominancia direccional  |
| `atr_ratio`          | volatilidad relativa    |
| `roc_10 / roc_20`    | aceleración             |
| `vol_zscore`         | expansión de interés    |
| `vol_ema_ratio`      | confirmación            |
| `body_range_ratio`   | intención               |
| `close_pos`          | cierre direccional      |
| `range_contraction`  | pre-expansión           |


In [47]:
# =========================================================
# 1️⃣ UNIVERSO BINANCE FUTURES
# =========================================================
def get_symbols():
    """
    Retorna todos los símbolos de futuros perpetuos USDT en Binance.
    """
    response = requests.get(
        f"{BINANCE_FUTURES_BASE}/fapi/v1/exchangeInfo",
        timeout=10
    )
    response.raise_for_status()

    data = response.json()

    symbols = []
    for s in data["symbols"]:
        if (
            s["quoteAsset"] == "USDT"
            and s["status"] == "TRADING"
            and s["contractType"] == "PERPETUAL"
            and not s["symbol"].endswith(("UPUSDT", "DOWNUSDT"))
            and not s["symbol"].startswith(("1000"))  # Excluir símbolos “multipliers” ya que no aportan para entrenamiento
        ):
            symbols.append(s["symbol"])
    
    return symbols



# =========================================================
# 2️⃣ DESCARGA VELAS HISTÓRICAS (15m)
# =========================================================
def fetch_klines(
    symbol: str,
    interval: str = "15m",
    limit: int = 1500,
    min_rows: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si min_rows > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    min_rows = min_rows or limit

    while len(all_data) < min_rows:
        fetch = min(1500, min_rows - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < min_rows:
        print(f"Insufficient data for {symbol}")
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["timestamp", "open", "high", "low", "close", "volume"]]
    df.sort_values("timestamp", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < min_rows:
        print(f"Insufficient data for {symbol}")
        return None

    return df



# =========================================================
# 3️⃣ INDICADORES BASE
# =========================================================
def compute_indicators(df):
    # EMAs base
    df["ema20"] = EMAIndicator(df["close"], 20).ema_indicator()
    df["ema50"] = EMAIndicator(df["close"], 50).ema_indicator()

    # ADX + DI
    adx = ADXIndicator(df["high"], df["low"], df["close"], 14)
    df["adx"] = adx.adx()
    df["di_plus"] = adx.adx_pos()
    df["di_minus"] = adx.adx_neg()

    # ATR
    atr = AverageTrueRange(df["high"], df["low"], df["close"], 14)
    df["atr"] = atr.average_true_range()

    return df

    

# =========================================================
# 4️ INDICADORES/FEATURES TÉCNICOS (SIMÉTRICOS LONG/SHORT)
# =========================================================
def slope(series, window=5):
    return series.diff(window) / window

import numpy as np

def compute_features(df):
    # =========================
    # Trend & Momentum
    # =========================
    df["ema_20_slope"] = slope(df["ema20"])
    df["ema_50_slope"] = slope(df["ema50"])

    df["ema_ratio_20_50"] = df["ema20"] / df["ema50"] - 1

    # =========================
    # Trend Strength
    # =========================
    # Se asume ADX, DI+ y DI- ya calculados
    # df["adx"]
    # df["di_plus"]
    # df["di_minus"]

    # =========================
    # Volatility
    # =========================
    df["atr_ratio"] = df["atr"] / df["close"]

    # =========================
    # Price Acceleration
    # =========================
    df["roc_10"] = df["close"].pct_change(10)
    df["roc_20"] = df["close"].pct_change(20)

    # =========================
    # Volume
    # =========================
    vol_mean = df["volume"].rolling(20).mean()
    vol_std = df["volume"].rolling(20).std()

    df["vol_zscore"] = (df["volume"] - vol_mean) / vol_std
    df["vol_ema_ratio"] = df["volume"] / df["volume"].ewm(span=20).mean()

    # =========================
    # Candle Structure
    # =========================
    body = (df["close"] - df["open"]).abs()
    rng = (df["high"] - df["low"]).replace(0, np.nan)

    df["body_range_ratio"] = body / rng
    df["close_pos"] = (df["close"] - df["low"]) / rng

    # =========================
    # Volatility Contraction (pre-breakout)
    # =========================
    atr_mean = df["atr"].rolling(20).mean()
    df["range_contraction"] = df["atr"] / atr_mean

    return df




# =========================================================
# 5️⃣ ALGORITMO DE ETIQUETADO (FORMAL)
# =========================================================
def label_continuations(df, horizon=4, alpha=1.0, beta=0.4):
    labels = []

    for i in range(len(df)):
        if i + horizon >= len(df):
            labels.append(np.nan)
            continue

        entry = df.loc[i, "close"]
        atr = df.loc[i, "atr"]

        tp_long = entry + alpha * atr
        sl_long = entry - beta * atr

        tp_short = entry - alpha * atr
        sl_short = entry + beta * atr

        future = df.iloc[i+1:i+1+horizon]

        label = 0

        # LONG
        if future["high"].max() >= tp_long:
            idx_tp = future[future["high"] >= tp_long].index[0]
            if future.loc[:idx_tp, "low"].min() > sl_long:
                label = 1

        # SHORT
        if future["low"].min() <= tp_short:
            idx_tp = future[future["low"] <= tp_short].index[0]
            if future.loc[:idx_tp, "high"].max() < sl_short:
                label = -1

        labels.append(label)

    df["target"] = labels
    return df



# =========================================================
# 6️⃣ CONSTRUCCIÓN FINAL DEL DATASET
# =========================================================
def build_dataset(symbols, train_size, test_size, folds, step_size):
    total_period = train_size + test_size + (folds * step_size)
    min_rows = int((total_period * 7 * 24 * 60) / 15)   # Cantidad mínima de velas de 15m requeridas para datasets de entrenamiento y test
    all_data = []

    for symbol in symbols:
        try:
            print(f"Descargando datos de entrenamiento para {symbol}")
            df = fetch_klines(symbol, interval="15m", limit=1500, min_rows=min_rows)

            if len(df) < min_rows:
                continue

            df = compute_indicators(df)
            df = df.dropna().reset_index(drop=True)

            df = compute_features(df)
            df = label_continuations(df)
            df["symbol"] = symbol

            df = df.dropna()

            if len(df) == 0:
                continue

            all_data.append(df)

        except Exception as e:
            print(f"Error en {symbol}: {e}")

    if not all_data:
        raise RuntimeError("No valid symbols produced data")

    return pd.concat(all_data, ignore_index=True)



# ============================================================
# 7️⃣ PIPELINE DE ENTRENAMIENTO ML + WALK-FORWARD VALIDATION
# Continuaciones LONG / SHORT (Binance, 15m)
# ============================================================

# ============================================================
# 7.1 CARGA Y PREPARACIÓN DEL DATASET
# ============================================================

def load_dataset(path):
    #df = pd.read_csv(path, parse_dates=["timestamp"])
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


def get_feature_columns(df):
    return [
        c for c in df.columns
        if c not in ["target", "symbol", "timestamp"]
    ]


# ============================================================
# 7.2 DEFINICIÓN DEL MODELO
# ============================================================

def build_model():
    return lgb.LGBMClassifier(
        boosting_type="gbdt",
        objective="binary",

        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0,

        n_estimators=800,
        learning_rate=0.03,

        max_depth=6,
        num_leaves=63,

        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,

        reg_alpha=0.5,
        reg_lambda=1.0,

        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )



# ============================================================
# 7.3 WALK-FORWARD SPLITS (VALIDACIÓN TEMPORAL)
# ============================================================

def walk_forward_splits(df, train_weeks, test_weeks, step_weeks):
    splits = []

    train_start = df["timestamp"].min()
    train_end = train_start + relativedelta(weeks=train_weeks)
    test_end = train_end + relativedelta(weeks=test_weeks)
    last_date = df["timestamp"].max()

    while test_end <= last_date:
  
        train_idx = df[
            (df["timestamp"] >= train_start) &
            (df["timestamp"] < train_end)
        ].index

        test_idx = df[
            (df["timestamp"] >= train_end) &
            (df["timestamp"] < test_end)
        ].index
        print("train_idx:", train_idx)
        print("test_idx:", test_idx)
        if len(train_idx) > 0 and len(test_idx) > 0:
            splits.append((train_idx, test_idx))

        train_start += relativedelta(weeks=step_weeks)
        train_end = train_start + relativedelta(weeks=train_weeks)
        test_end = train_end + relativedelta(weeks=test_weeks)
        print("train_start:", train_start)
        print("train_end:", train_end)
        print("test_end:", test_end)
        
    return splits


# ============================================================
# 7.4 PIPELINE DE ENTRENAMIENTO + EVALUACIÓN
# ============================================================

def train_and_evaluate(df):
    splits = walk_forward_splits(df, TRAIN_WEEKS, TEST_WEEKS, STEP)

    all_metrics = []

    for i, (train_idx, test_idx) in enumerate(splits, 1):
        print(f"\n===== SPLIT {i} =====")

        train = df.loc[train_idx]
        test = df.loc[test_idx]

        # --- Balanceo inteligente ---
        pos = train[train[TARGET] == 1]
        neg = train[train[TARGET] == 0].sample(
            frac=NEG_DOWNSAMPLE_FRAC, random_state=42
        )

        train_bal = pd.concat([pos, neg]).sample(frac=1, random_state=42)

        X_train = train_bal[FEATURES]
        y_train = train_bal[TARGET]

        X_test = test[FEATURES]
        y_test = test[TARGET]

        # --- Entrenamiento ---
        model = build_model()
        model.fit(X_train, y_train)

        # --- Inferencia ---
        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba >= PROBA_THRESHOLD).astype(int)

        # --- Métricas operativas ---
        mask = y_test != 0
        precision = precision_score(
            y_test[mask],
            preds[mask],
            average="macro",
            zero_division=0
        )

        if mask.sum() > 0:
            cm_trades = confusion_matrix(
                y_test[mask],
                preds[mask],
                labels=[-1, 1]
            )
            tn, fp, fn, tp = cm_trades.ravel()
        else:
            tn = fp = fn = tp = 0

        trade_rate = (tp + fp) / len(y_test)
        fp_tp_ratio = fp / max(tp, 1)

        metrics = {
            "split": i,
            "precision": round(precision, 4),
            "tp": int(tp),
            "fp": int(fp),
            "fp_tp_ratio": round(fp_tp_ratio, 3),
            "trade_rate": round(trade_rate, 4)
        }

        all_metrics.append(metrics)

        print(metrics)

    return pd.DataFrame(all_metrics)



# ============================================================
# 8️⃣ MAIN DE ENTRENAMIENTO
# ============================================================

def train_main(data_source, file_path):
    if data_source == "FILE":
        print("\nCargando dataset...")
        df = load_dataset(file_path)
    else:
        symbols = get_symbols()
        df = build_dataset(symbols, train_size=TRAIN_WEEKS, test_size=TEST_WEEKS, folds=FOLDS, step_size=STEP)

        current_date = str(datetime.now())[:16].replace(":", "").replace(" ", "_")
        dataset_filename = 'Scaler_continuacion_futuros_cripto_SPLIT' + str(TRAIN_WEEKS) + "-" + str(TEST_WEEKS) + "_" + current_date + '.parquet'
        dataset_filepath = "C:/Users/Lenovo S540/Python_scripts/DATASETS_FUTURES_CONTINUATION/" + dataset_filename
        #df.to_csv(dataset_filepath, index=False)
        df.to_parquet(dataset_filepath, engine='pyarrow', compression='snappy')
        print("Dataset generado:", df.shape)

    print("Distribución de clases:")
    print(df["target"].value_counts(normalize=True))

    results_df = train_and_evaluate(df)

    print("\n===== RESUMEN GLOBAL =====")
    print(results_df.mean(numeric_only=True))
    print("\nDesviación estándar:")
    print(results_df.std(numeric_only=True))   

    final_model = LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    final_model.fit(df[FEATURES], df[TARGET])
    
    joblib.dump(final_model, "model_continuaciones_lgb.pkl")


    print("\nPipeline completado con éxito.")    

    return results_df, final_model
    

In [4]:
# =========================================================
# EJECUCIÓN
# =========================================================
print("🧠 Entrenando modelo…")
results_df, final_model = train_main(data_source = DATA_SOURCE, file_path = DATASET_PATH)

🧠 Entrenando modelo…
Descargando datos de entrenamiento para BTCUSDT
Descargando datos de entrenamiento para ETHUSDT
Descargando datos de entrenamiento para BCHUSDT
Descargando datos de entrenamiento para XRPUSDT
Descargando datos de entrenamiento para LTCUSDT
Descargando datos de entrenamiento para TRXUSDT
Descargando datos de entrenamiento para ETCUSDT
Descargando datos de entrenamiento para LINKUSDT
Descargando datos de entrenamiento para XLMUSDT
Descargando datos de entrenamiento para ADAUSDT
Descargando datos de entrenamiento para XMRUSDT
Descargando datos de entrenamiento para DASHUSDT
Descargando datos de entrenamiento para ZECUSDT
Descargando datos de entrenamiento para XTZUSDT
Descargando datos de entrenamiento para BNBUSDT
Descargando datos de entrenamiento para ATOMUSDT
Descargando datos de entrenamiento para ONTUSDT
Descargando datos de entrenamiento para IOTAUSDT
Descargando datos de entrenamiento para BATUSDT
Descargando datos de entrenamiento para VETUSDT
Descargando dat

In [5]:
results_df

,split,precision,tp,fp,fp_tp_ratio,trade_rate
0,1,0.2049,991,621,0.627,0.0025
1,2,0.2049,632,396,0.627,0.0016
2,3,0.1971,566,391,0.691,0.0015
3,4,0.2204,898,460,0.512,0.0021
4,5,0.2190,657,343,0.522,0.0015


## Guardar Modelo y Scaler en Disco Duro

In [6]:
current_date = str(datetime.now())[:16].replace(":", "").replace(" ", "_")
model_filename = 'Modelo_continuacion_futuros_cripto_SPLIT' + str(TRAIN_WEEKS) + "-" + str(TEST_WEEKS) + "_" + current_date + '.joblib'

joblib.dump(final_model, model_filename)
print(f"Model saved successfully as {model_filename}")


Model saved successfully as Modelo_continuacion_futuros_cripto_SPLIT6-2_2026-02-08_1847.joblib


# 1. EJECUCIÓN DEL MODELO (INFERENCIA)

## Flujo operativo final

scanner → lista de setups POTENCIALES
                   
                   ↓
           trader selecciona símbolo
                   
                   ↓
        monitor 5m durante X minutos
                   
                   ↓
            alerta SOLO si hay entrada limpia



## Cargar Modelo anterior

In [48]:
model_filename = "Modelo_continuacion_futuros_cripto_SPLIT6-2_2026-02-08_1847.joblib"

final_model = joblib.load(model_filename)
print(f"Modelo {model_filename} cargado exitosamente")

Modelo Modelo_continuacion_futuros_cripto_SPLIT6-2_2026-02-08_1847.joblib cargado exitosamente


In [5]:
final_model

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
               num_class=3, num_leaves=63, objective='multiclass',
               random_state=42, subsample=0.8)

## Generar Inferencias

¿Hacia dónde es más probable la reversión? → Dir_Confidence: Claridad direccional de la reversión, confianza relativa (0-1)


¿Cuánto puede moverse en términos absolutos? → ATR: Cuánto se mueve este activo en promedio por vela de 1h en USDT                                                                                                  

< 0.55	Señal ambigua → NO operar

0.55 – 0.65	Reversión posible, timing crítico

0.65 – 0.75	Reversión clara

'> 0.75	Reversión muy bien alineada

¿Qué tan explotable es ese movimiento en términos relativos? → ATR_rel: Qué % del precio puede moverse por hora

In [49]:
def build_universe():
    """
    Construye el universo de símbolos líquidos y estables
    """
    tickers = requests.get(
        f"{BINANCE_FUTURES_BASE}/fapi/v1/ticker/24hr"
    ).json()

    symbols = []

    for t in tickers:
        symbol = t["symbol"]
        if not symbol.endswith("USDT"):
            continue
        if float(t["quoteVolume"]) < 20_000_000:
            continue

        symbols.append(symbol)

    return symbols



def build_features_15m(df):
    """
    Construye EXACTAMENTE las mismas features usadas en entrenamiento
    para la última vela disponible (15m)
    """

    # Indicadores base
    df = compute_indicators(df)
    #print(f"df indicators: {df.shape}")
    
    # Features de modelo
    df = compute_features(df)
    #print(f"df features: {df.info()}")
    
    # Limpieza
    df = df.dropna(subset=FEATURES)
    #print(f"df final features: {df.shape}")
    
    if df.empty:
        return None

    # Tomar última fila y mismas columnas
    X = df[FEATURES].iloc[-1:]

    return X



def scan_15m(symbols, model):
    """
    Evalúa continuaciones en timeframe 15m
    """
    signals = []

    for symbol in symbols:
        try:
            df = fetch_klines(symbol, interval="15m", limit=500, min_rows=LOOKBACK_15M)

            X = build_features_15m(df)

            proba = model.predict_proba(X)[0]

            pred = np.argmax(proba) - 1
            conf = np.max(proba)
            
            if pred != 0 and conf >= CONFIDENCE_THRESHOLD:
                signals.append({
                    "symbol": symbol,
                    "side": "LONG" if pred == 1 else "SHORT",   # 0: LATERAL; 1: LONG; -1: SHORT
                    "confidence": conf,
                    "price": df.close.iloc[-1],
                    "distribution": proba,
                    "time": datetime.now()
                })

        except Exception as e:
            print(f"[15m] {symbol}: {e}")

    if len(signals) > 0:
        signals = pd.DataFrame(signals).sort_values(by='confidence', ascending=False)
        signals = signals.reset_index(drop=True)
        local_warning(signals)
        
    return signals



def local_warning(df: pd.DataFrame()):
    if df is not None:
    
        frequency = 2500  # Set Frequency To 2500 Hertz
        duration = 1000   # Set Duration To 1000 ms == 1 second
    
        winsound.Beep(frequency, duration)
        print("Oportunidades de entrada encontradas!")
        
        # Example of beeping with time delays
        for i in range(3):
            winsound.Beep(500, 200) # Frequency 500 Hz, duration 200ms
            time.sleep(0.5)        # Pause for 0.5 seconds between beeps

In [50]:
print("🔍 Filtrando símbolos…")
symbols = build_universe()
print(f"Total símbolos USDT Futures: {len(symbols)}")

print("🔍 Escaneando mercado…")
signals = scan_15m(symbols, final_model)


🔍 Filtrando símbolos…
Total símbolos USDT Futures: 132
🔍 Escaneando mercado…
[15m] FTMUSDT: Expected 2D array, got scalar array instead:
array=None.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
[15m] VIDTUSDT: Expected 2D array, got scalar array instead:
array=None.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
[15m] STRAXUSDT: Expected 2D array, got scalar array instead:
array=None.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
[15m] MKRUSDT: Expected 2D array, got scalar array instead:
array=None.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
[15m] MEMEFIUSDT: Expected 2D array, got scalar array instead:
a

In [51]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)
signals

[]

# 2. CONFIRMACIÓN DE ENTRADA DE LA ORDEN

## Confirmación de entrada 5m

1. Universe Builder      → qué símbolos miro (Reducir ruido, Aumentar base rate, Evitar microcaps y símbolos erráticos)

2. Features Builder      → (dónde hay posible continuación (Capturar direccionalidad + aceleración, Evitar features, frágiles), Totalmente causal (sin lookahead))

3. 15m Scanner (ML)      → Detectar contexto favorable, NO decide entrada, Reduce mercado a oportunidades reales

4. 5m Entry Monitor      → cuándo entrar (timing fino) (Evitar:entradas tardías, pullbacks profundos, rompimientos sin volumen)

5. Trade Manager         → SL, TP, BE, trailing (luego)


In [52]:
TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")


In [53]:
def confirm_entry_5m(
    signal,
    vol_mult=VOL_MULT,
    monitor_minutes=30,
    poll_seconds=60
):
    """
    Monitorea entrada fina en 5m durante un periodo definido.
    Si se cumplen condiciones de continuación, envía alerta Telegram.
    """

    symbol = signal["symbol"]
    side = signal["side"]

    end_time = datetime.utcnow() + timedelta(minutes=monitor_minutes)

    msg = f"[5m] Monitoring {symbol} {side} for {monitor_minutes} minutes at {datetime.now()}"
    send_telegram_alert(msg)
    print(msg)

    while datetime.utcnow() < end_time:

        df = fetch_klines(
            symbol,
            interval="5m",
            limit=200,
            min_rows=LOOKBACK_5M
        )

        if df is None or df.empty:
            time.sleep(poll_seconds)
            continue

        # Indicadores de estructura
        df["ema20"] = df.close.ewm(span=20).mean()
        df["ema50"] = df.close.ewm(span=50).mean()

        last = df.iloc[-1]
        prev = df.iloc[-2]

        # Volumen relativo
        vol_mean = df.volume.rolling(20).mean().iloc[-1]
        vol_ok = last.volume > vol_mult * vol_mean if vol_mean > 0 else False

        entry_ok = False
        print(f"vol_ok check at {datetime.now()}: {vol_ok}")

        if side == "LONG":
            print(f"CONDICION 1: {last.close} > {last.ema20} > {last.ema50}")
            print(f"CONDICION 2: {prev.close} <= {prev.ema20}\n")
            entry_ok = (
                last.close > last.ema20 > last.ema50 and
                prev.close <= prev.ema20 and  # rompe estructura
                vol_ok
            )

        elif side == "SHORT":
            print(f"CONDICION 1: {last.close} < {last.ema20} < {last.ema50}")
            print(f"CONDICION 2: {prev.close} >= {prev.ema20}\n")
            entry_ok = (
                last.close < last.ema20 < last.ema50 and
                prev.close >= prev.ema20 and
                vol_ok
            )

        if entry_ok:
            msg = f"""
            🚀 *CONTINUACIÓN DETECTADA*
            
            *Símbolo:* `{signal['symbol']}`
            *Dirección:* `{signal['side']}`
            *Confianza:* `{signal['confidence']:.2f}`
            *Price:* `{last.close:.4f}\n`
            *Volume `{last.volume}\n`
            ⏱ Timeframe señal: 15m  
            🎯 Entrada confirmada en 5m
            ⚠️ *Esperar gestión de riesgo*
            """
            send_telegram_alert(msg)
            print(msg)

            return True

        time.sleep(poll_seconds)

    msg = f"[5m] Monitoring expired for {symbol}"
    send_telegram_alert(msg)
    print(msg)
    
    return False


In [40]:
signal

symbol                                                               TAOUSDT
side                                                                   SHORT
confidence                                                          0.414379
price                                                                 157.36
distribution    [0.4143794844255514, 0.3826482122026425, 0.2029723033718061]
time                                              2026-02-10 10:50:49.026644
Name: 0, dtype: object

In [54]:
posicion = 0

# signal = signals.loc[posicion]
# symbol = signals.loc[posicion, "symbol"]
# side = signals.loc[posicion, "side"]

signal["symbol"] = "NILUSDT"
signal["side"] = "LONG"


# print("symbol:", symbol)
# print("side:", side)

In [55]:
confirm_entry_5m(signal)

[5m] Monitoring NILUSDT LONG for 30 minutes at 2026-02-11 09:57:44.628585
vol_ok check at 2026-02-11 09:57:45.770107: False
CONDICION 1: 0.05571 > 0.0568638307728622 > 0.05795354801121629
CONDICION 2: 0.05619 <= 0.05698561863604061

vol_ok check at 2026-02-11 09:58:46.101207: False
CONDICION 1: 0.05587 > 0.05687910654381599 > 0.05796044830084254
CONDICION 2: 0.05619 <= 0.05698561863604061

vol_ok check at 2026-02-11 09:59:46.464116: False
CONDICION 1: 0.05554 > 0.0568476002662238 > 0.05794621645348841
CONDICION 2: 0.05619 <= 0.05698561863604061

vol_ok check at 2026-02-11 10:00:46.739457: False
CONDICION 1: 0.0552 > 0.05668590265227031 > 0.05781480552936144
CONDICION 2: 0.0555 <= 0.056842740997344286

vol_ok check at 2026-02-11 10:01:47.016545: False
CONDICION 1: 0.0549 > 0.05665726058173194 > 0.05780186748631223
CONDICION 2: 0.0555 <= 0.056842740997344286

vol_ok check at 2026-02-11 10:02:47.332302: False
CONDICION 1: 0.05512 > 0.05667826476679342 > 0.05781135538454832
CONDICION 2: 0.

False

# 3. CONFIRMACIÓN DE SALIDA DE LA ORDEN

## Confirmación de salida 15m

In [20]:
class Trade:
    def __init__(self, symbol, side, entry_price, atr_15m):
        self.symbol = symbol
        self.side = side
        self.entry = entry_price
        self.atr = atr_15m

        self.sl = self._init_sl()
        self.tp1 = self._tp1()
        self.tp2 = self._tp2()

        self.tp1_hit = False
        self.active = True

    def _init_sl(self):
        if self.side == "LONG":
            return self.entry - SL_ATR * self.atr
        return self.entry + SL_ATR * self.atr

    def _tp1(self):
        if self.side == "LONG":
            return self.entry + TP1_ATR * self.atr
        return self.entry - TP1_ATR * self.atr

    def _tp2(self):
        if self.side == "LONG":
            return self.entry + TP2_ATR * self.atr
        return self.entry - TP2_ATR * self.atr


In [22]:
def manage_trade(trade):
    """
    Gestiona un trade hasta cierre
    """
    msg = f"📥 *TRADE ABIERTO*\n\n \
        Time: {datetime.now()}\n \
        {trade.symbol} | {trade.side}\n \
        Entry: {trade.entry}\n \
        SL: {trade.sl}\n \
        TP1: {trade.tp1}\n \
        TP2: {trade.tp2}"
    send_telegram_alert(msg)
    print(msg)

    while trade.active:
        try:
            df = fetch_klines(trade.symbol, interval="5m", limit=100, min_rows=50)
            df["ema20"] = df.close.ewm(span=20).mean()
            last_price = df.close.iloc[-1]

            # STOP LOSS
            if trade.side == "LONG" and last_price <= trade.sl:
                trade.active = False
                msg = f"📤 *TRADE CERRADO*\n\n \
                    Time: {datetime.now()}\n \
                    {trade.symbol} | {trade.side}\n \
                    Razón: STOP LOSS ALCANZADO\n \
                    Precio: {last_price}"
                send_telegram_alert(msg)
                print(msg)
                break

            if trade.side == "SHORT" and last_price >= trade.sl:
                trade.active = False
                msg = f"📤 *TRADE CERRADO*\n\n \
                    Time: {datetime.now()}\n \
                    {trade.symbol} | {trade.side}\n \
                    Razón: STOP LOSS ALCANZADO\n \
                    Precio: {last_price}"
                send_telegram_alert(msg)
                print(msg)
                break

            # TP1
            if not trade.tp1_hit:
                if (
                    trade.side == "LONG" and last_price >= trade.tp1
                ) or (
                    trade.side == "SHORT" and last_price <= trade.tp1
                ):
                    trade.tp1_hit = True
                    trade.sl = trade.entry  # BE
                    msg = f"🎯 *TP1 ALCANZADO*\n\n \
                        Time: {datetime.now()}\n \
                        {trade.symbol} | {trade.side}\n \
                        SL movido a BE: {trade.entry}"
                    send_telegram_alert(msg)
                    print(msg)

            # TP2
            if trade.tp1_hit:
                if (
                    trade.side == "LONG" and last_price >= trade.tp2
                ) or (
                    trade.side == "SHORT" and last_price <= trade.tp2
                ):
                    #trade.active = False
                    msg = f"🎯 *TP2 ALCANZADO*\n\n \
                        Time: {datetime.now()}\n \
                        {trade.symbol} | {trade.side}"
                    send_telegram_alert(msg)
                    print(msg)
                    break

                # TRAILING
                ema20 = df.ema20.iloc[-1]
                actual_sl = trade.sl
                
                if trade.side == "LONG":
                    trade.sl = max(
                        trade.sl,
                        ema20 - TRAIL_BUFFER_ATR * trade.atr
                    )
                    
                else:
                    trade.sl = min(
                        trade.sl,
                        ema20 + TRAIL_BUFFER_ATR * trade.atr
                    )

                if actual_sl != trade.sl:
                    msg = f"📤 *TRAILING STOP SUGERIDO*\n\n \
                        Time: {datetime.now()}\n \
                        {trade.symbol} | {trade.side}\n \
                        Nuevo SL: {trade.sl}\n \
                        Precio: {last_price}"
                    send_telegram_alert(msg)
                    print(msg)

        except Exception as e:
            msg = f"[TradeManager] {trade.symbol}: {e}"
            send_telegram_alert(msg)
            print(msg)

        time.sleep(TRADE_CHECK_INTERVAL)


In [25]:
#symbol = "ZECUSDT"
#side = "SHORT"

entry_price = fetch_klines(symbol, interval="5m", limit=50, min_rows=1).close.iloc[-1]
 
df_15m = fetch_klines(symbol, interval="15m", limit=100, min_rows=50)
df_15m = compute_indicators(df_15m)
atr_15m = df_15m["atr"].iloc[-1]



trade = Trade(symbol, side, entry_price, atr_15m)
manage_trade(trade)


📥 *TRADE ABIERTO*

         Time: 2026-02-09 11:05:12.500861
         HYPEUSDT | SHORT
         Entry: 31.337
         SL: 31.686380388749814
         TP1: 31.045849676041822
         TP2: 30.609124190104556
🎯 *TP1 ALCANZADO*

                         Time: 2026-02-09 11:09:15.728313
                         HYPEUSDT | SHORT
                         SL movido a BE: 31.337
📤 *TRAILING STOP SUGERIDO*

                     Time: 2026-02-09 11:09:16.468720
                     HYPEUSDT | SHORT
                     Nuevo SL: 31.337
                     Precio: 30.976
📤 *TRAILING STOP SUGERIDO*

                     Time: 2026-02-09 11:09:47.459179
                     HYPEUSDT | SHORT
                     Nuevo SL: 31.337
                     Precio: 30.984
📤 *TRAILING STOP SUGERIDO*

                     Time: 2026-02-09 11:10:18.396999
                     HYPEUSDT | SHORT
                     Nuevo SL: 31.337
                     Precio: 30.978
📤 *TRAILING STOP SUGERIDO*

               